In [0]:
from pyspark.sql import functions as F

bronze_match_player_details = spark.read.table("opendota.bronze.match_player_details")

CHECKPOINT_PATH = "/Volumes/opendota/silver/_checkpoints/match_player_details"


def transform_match_player_details(df):
    return (
        df
        .drop(
            "__index_level_0__", 
            "_airbyte_meta", 
            "_airbyte_raw_id", 
            "_airbyte_extracted_at", 
            "_airbyte_generation_id", 
            "_ab_source_file_url", 
            "_ab_source_file_last_modified",
            "last_login",
            "is_subscriber",
            "is_contributor",
            "pred_vict",
            "name",
            "party_id",
            "party_size",
            "team_number",
            "team_slot",
            "moonshard",
            "aghanims_scepter", 
            "aghanims_shard",
            "item_0", 
            "item_1", 
            "item_2", 
            "item_3", 
            "item_4", 
            "item_5",
            "backpack_0",
            "backpack_1",
            "backpack_2",
            "item_neutral",
            "item_neutral2",
            "multi_kills",
            "kill_streaks",
            "hero_kills",
            "tower_kills",
            "roshan_kills",
            "observers_placed",
            "computed_mmr"
        )
        .withColumnsRenamed({
            "isRadiant": "is_radiant",
            "personaname": "persona_name"
        })
        .withColumns({
            "start_time": F.from_unixtime(F.col("start_time")).cast("timestamp"),
            "kda": F.col("kda").cast("double"),
            "stuns": F.col("stuns").cast("double"),
            "rank_tier": F.col("rank_tier").cast("double"),
            "kills_per_min": F.col("kills_per_min").cast("double"),
            "lane_efficiency": F.col("lane_efficiency").cast("double"),
            "teamfight_participation": F.col("teamfight_participation").cast("double"),
            "lane_efficiency_pct": F.col("lane_efficiency_pct").cast("double"),
            "gold_per_min": F.col("gold_per_min").cast("int"),
            "xp_per_min": F.col("xp_per_min").cast("int"),
            "net_worth": F.col("net_worth").cast("int"),
            "gold_spent": F.col("gold_spent").cast("int"),
            "total_gold": F.col("total_gold").cast("int"),
            "total_xp": F.col("total_xp").cast("int"),
            "hero_damage": F.col("hero_damage").cast("int"),
            "tower_damage": F.col("tower_damage").cast("int"),
            "hero_healing": F.col("hero_healing").cast("int"),
            "gold": F.col("gold").cast("int"),
            "duration": F.col("duration").cast("int"),
            "kills": F.col("kills").cast("smallint"),
            "life_state_dead": F.col("life_state_dead").cast("int"),
            "deaths": F.col("deaths").cast("smallint"),
            "assists": F.col("assists").cast("smallint"),
            "denies": F.col("denies").cast("smallint"),
            "last_hits": F.col("last_hits").cast("smallint"),
            "obs_placed": F.col("obs_placed").cast("smallint"),
            "sen_placed": F.col("sen_placed").cast("smallint"),
            "rune_pickups": F.col("rune_pickups").cast("smallint"),
            "sentry_uses": F.col("sentry_uses").cast("smallint"),
            "observer_uses": F.col("observer_uses").cast("smallint"),
            "sentry_kills": F.col("sentry_kills").cast("smallint"),
            "observer_kills": F.col("observer_kills").cast("smallint"),
            "ancient_kills": F.col("ancient_kills").cast("smallint"),
            "buyback_count": F.col("buyback_count").cast("smallint"),
            "camps_stacked": F.col("camps_stacked").cast("smallint"),
            "courier_kills": F.col("courier_kills").cast("smallint"),
            "neutral_kills": F.col("neutral_kills").cast("smallint"),
            "towers_killed": F.col("towers_killed").cast("smallint"),
            "roshans_killed": F.col("roshans_killed").cast("smallint"),
            "actions_per_min": F.col("actions_per_min").cast("smallint"),
            "necronomicon_kills": F.col("necronomicon_kills").cast("smallint"),
            "abandons": F.col("abandons").cast("smallint"),
            "creeps_stacked": F.col("creeps_stacked").cast("smallint"),
            "lane_kills": F.col("lane_kills").cast("smallint"),
            "hero_id": F.col("hero_id").cast("smallint"),
            "hero_variant": F.col("hero_variant").cast("smallint"),
            "patch": F.col("patch").cast("smallint"),
            "region": F.col("region").cast("smallint"),
            "cluster": F.col("cluster").cast("smallint"),
            "game_mode": F.col("game_mode").cast("smallint"),
            "lobby_type": F.col("lobby_type").cast("smallint"),
            "player_slot": F.col("player_slot").cast("smallint"),
            "lane": F.col("lane").cast("tinyint"),
            "lane_role": F.col("lane_role").cast("tinyint"),
            "leaver_status": F.col("leaver_status").cast("tinyint"),
            "level": F.col("level").cast("tinyint"),
            "firstblood_claimed": F.col("firstblood_claimed").cast("boolean"),
            "win": F.col("win").cast("boolean"),
            "lose": F.col("lose").cast("boolean"),
            "processed_at": F.current_timestamp(),
        })
    .withColumn("match_date", F.to_date("start_time"))
    )


def process_batch(df_batch, batch_id):
    transformed_df = transform_match_player_details(df_batch)

    (
        transformed_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable("opendota.silver.match_player_details")
    )


(
    spark.readStream
    .table("opendota.bronze.match_player_details")
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)